# Session 3 — Cutting Text Into Pieces

### Study Buddy · Session 3 of 8

Last session we worked out what we need: **send only the pieces of the documents that are
relevant to this question.**

Before we can find pieces, we need pieces. That's today. It sounds like the boring
housekeeping bit, and it is the single most under-rated part of the whole technique — a
badly cut document cannot be rescued by anything downstream.

## TODAY:

- **Part A:** loading a folder of documents properly
- **Part B:** splitting, and why the size matters so much
- **Part C:** the boundary problem, and overlap

In [ ]:
# imports

import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Setup

load_dotenv(override=True)

DOCUMENTS_DIR = "documents"

## PART A: Loading documents properly

Last session we read files by hand with `Path.read_text()` and glued them into one string.
That worked, but we lost something important: **once it's all one string, you no longer
know which file anything came from.**

And we're going to need that. When the assistant answers, we want it to be able to say
*"according to hardware.md"*. You can't cite a source you've thrown away.

LangChain's answer to this is an object called a **`Document`**: a piece of text, plus a
dictionary of facts about where it came from. We'll build them ourselves, because it's four
lines and you should be able to see exactly what's in one.

In [ ]:
def load_documents(folder):
    """
    Read every .md file in a folder and return them as LangChain Documents.

    A Document is just text + metadata. We build the metadata ourselves here, which means
    we can put anything useful in it - and it will travel with the text all the way
    through splitting, embedding and retrieval.

    encoding="utf-8" is not optional: without it, Windows sometimes guesses a different
    encoding and mangles every non-English character in your notes.
    """
    documents = []
    # sorted() so the order is the same on every machine and every run. Without it the
    # order depends on the filesystem, and "it worked yesterday" is the worst kind of bug.
    for path in sorted(Path(folder).glob("**/*.md")):
        text = path.read_text(encoding="utf-8")
        documents.append(Document(page_content=text, metadata={"filename": path.name}))
    return documents


documents = load_documents(DOCUMENTS_DIR)

print(f"Loaded {len(documents)} documents")

### What is a Document?

Don't guess — look.

In [ ]:
documents[0]

In [ ]:
# It has exactly two parts. First, the text:

print(documents[0].page_content[:300])

In [ ]:
# And second, everything we know about it:

documents[0].metadata

That `metadata` dictionary is the whole reason we bothered with `Document` instead of just
gluing all the text together like we did last session. It travels with the text everywhere
it goes — through splitting, through embedding, through retrieval — so at the very end we
can still say which file an answer came from.

In [ ]:
# All of them at once.

for doc in documents:
    print(f"{doc.metadata['filename']:20} {len(doc.page_content):>7,} characters")

### Adding your own metadata

You can put anything you like in there, and it'll come back to you at retrieval time.

In [ ]:
# Add the size, just to prove metadata is ours to shape.

for doc in documents:
    doc.metadata["characters"] = len(doc.page_content)

documents[0].metadata

<div style="border-left: 6px solid #181; background: #f3fbf3; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#181; margin-top:0;">🌍 Where you'll see this in the real world</h3>
<p style="color:#181; margin-bottom:0;">
Real systems put a lot into metadata: which department wrote it, when it was last updated,
who's allowed to see it, which version of the product it describes.<br/><br/>
Then they <b>filter on it</b>: "only search documents this user is allowed to read", "only
search the current version of the manual". That's how a company chatbot avoids telling a
customer about an internal policy, and it's all just this dictionary.
</p>
</div>

## PART B: Splitting — and why size matters

Now we cut. LangChain's `RecursiveCharacterTextSplitter` is the standard tool, and the
"recursive" in the name describes something genuinely clever.

It tries to split on paragraph breaks first. If a piece is still too big, it tries single
line breaks. Then sentences. Then words. Then, only as a last resort, in the middle of a
word.

So it breaks at the **most natural place available** rather than blindly chopping every N
characters.

In [ ]:
# Let's try a very small chunk size first, so the effect is obvious.

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)
tiny_chunks = splitter.split_documents(documents)

print(f"{len(documents)} documents became {len(tiny_chunks)} chunks")

In [ ]:
# What does a 200-character chunk actually look like? Read three of them.

for chunk in tiny_chunks[10:13]:
    print(f"--- from {chunk.metadata['filename']} ---")
    print(chunk.page_content)
    print()

### Now the other extreme

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=0)
huge_chunks = splitter.split_documents(documents)

print(f"{len(documents)} documents became {len(huge_chunks)} chunks")
print(f"\nThat's barely a split at all - our files are about that size to begin with.")

### The trade-off, stated plainly

| Chunk size | What goes wrong |
|---|---|
| **Too small** (~200) | A chunk stops mid-thought. "The two-tracker rule was added after" — added after *what*? The answer was in the next chunk, which didn't come back. |
| **Too large** (~4000) | You're back to Session 2's problem in miniature. The chunk that comes back is mostly irrelevant text, and you're paying for all of it. |
| **About right** (500-1000) | Roughly one idea per chunk. Big enough to stand alone, small enough to be mostly about one thing. |

There is no universally correct number. It depends on your documents — dense reference
notes want smaller chunks, flowing prose wants larger ones. **500 to 1000 is where you
start**, and then you measure.

In [ ]:
# Compare four sizes at once. Notice how the chunk count changes.

for size in [200, 500, 1000, 4000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0)
    chunks = splitter.split_documents(documents)
    average = sum(len(c.page_content) for c in chunks) / len(chunks)
    print(f"chunk_size={size:<6} → {len(chunks):>3} chunks, average {average:>6.0f} characters")

Notice the averages are always *below* the chunk size you asked for. That's the splitter
preferring a natural break over filling the chunk to the brim — which is exactly what you
want it to do.

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself — the judgement call</h3>
<p style="color:#900; margin-bottom:0;">
Run the cell below, which prints one chunk from each of the four sizes.<br/><br/>
Then answer this honestly: <b>imagine you were handed only that chunk and asked "why does
Kestrel carry two trackers?" — could you answer from it?</b><br/><br/>
That is the only test that matters, and it's a judgement, not a formula. You are the one
who decides.
</p>
</div>

In [ ]:
# The same part of the same document, cut four different ways.

for size in [200, 500, 1000, 4000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0)
    chunks = splitter.split_documents(documents)

    # Find the first chunk that mentions trackers
    tracker_chunks = [c for c in chunks if "tracker" in c.page_content.lower()]

    print("=" * 70)
    print(f"chunk_size = {size}")
    print("=" * 70)
    print(tracker_chunks[0].page_content if tracker_chunks else "(none found)")
    print()

## PART C: The boundary problem, and overlap

Here's a problem you cannot avoid by choosing a better size.

Wherever you cut, you cut *somewhere*. And sometimes the cut lands in the middle of the
one sentence that answers the question — so half the answer is in chunk 7 and half is in
chunk 8, and neither chunk on its own says anything useful.

In [ ]:
# Let's deliberately go looking for a bad cut.

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
chunks = splitter.split_documents(documents)

# Print the end of one chunk and the start of the next, to see the seam
for i in range(5, 8):
    print(f"--- end of chunk {i} ---")
    print("..." + chunks[i].page_content[-120:])
    print(f"--- start of chunk {i + 1} ---")
    print(chunks[i + 1].page_content[:120] + "...")
    print("=" * 70)

### The fix: overlap

Let each chunk include a bit of the end of the previous one. Then anything that falls on a
boundary appears **whole** in at least one chunk.

You pay for it in storage and a little duplication. It is, universally, worth it.

In [ ]:
# Same split, with overlap

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
overlapped = splitter.split_documents(documents)

print(f"Without overlap: {len(chunks)} chunks")
print(f"With overlap:    {len(overlapped)} chunks  (more chunks, because they repeat a bit)")

In [ ]:
# Look at the seam now - the end of one chunk reappears at the start of the next

print("--- end of chunk 5 ---")
print("..." + overlapped[5].page_content[-150:])
print("\n--- start of chunk 6 ---")
print(overlapped[6].page_content[:150] + "...")

### How much overlap?

**About 10-20% of the chunk size.** With `chunk_size=1000`, use `chunk_overlap=150` or `200`.

Less, and boundary-straddling sentences still get lost. More, and you're mostly storing the
same text repeatedly, which costs space and makes near-identical chunks compete with each
other in search results.

In [ ]:
# Our settings for the rest of the course. These are the numbers we'll carry into
# the real project in Session 6.

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(documents)

print(f"{len(documents)} documents → {len(chunks)} chunks")
print(f"Average chunk: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")

In [ ]:
# Every chunk still knows where it came from. This is the payoff from Part A.

for chunk in chunks[:5]:
    preview = chunk.page_content[:60].replace("\n", " ")
    print(f"{chunk.metadata['filename']:15} | {preview}...")

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Point the loader at <b>your own</b> documents and split them.<br/><br/>
Then: read <b>five random chunks</b> of your own material. For each one, ask "if this were
the only thing I was given, what questions could I answer?"<br/><br/>
Write down the chunk size and overlap you'll use for your material, <b>and one sentence
saying why.</b> You'll need this in Session 6, and "because it was the default" is not an
acceptable answer.
</p>
</div>

In [ ]:
# Your documents, your settings.

MY_CHUNK_SIZE = 1000
MY_CHUNK_OVERLAP = 200

my_documents = load_documents("documents")

my_splitter = RecursiveCharacterTextSplitter(
    chunk_size=MY_CHUNK_SIZE, chunk_overlap=MY_CHUNK_OVERLAP
)
my_chunks = my_splitter.split_documents(my_documents)

print(f"{len(my_documents)} documents → {len(my_chunks)} chunks")

In [ ]:
# Read five random ones. random.sample picks without repeats.

import random

for chunk in random.sample(my_chunks, min(5, len(my_chunks))):
    print("=" * 70)
    print(chunk.page_content)
    print()

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Extra challenge</h3>
<p style="color:#f71; margin-bottom:0;">
Our documents have markdown headings (<code>## Trackers</code>). The
<code>RecursiveCharacterTextSplitter</code> doesn't know or care — it treats them as
ordinary text.<br/><br/>
LangChain has a <code>MarkdownHeaderTextSplitter</code> that splits <i>by heading</i> and
records the heading in the metadata, so every chunk knows which section it came from.<br/><br/>
Try it. Is it better for our documents? Would it be better for yours? What happens if a
single section is 10,000 characters long?
</p>
</div>

## Where we got to

- A `Document` is text + metadata, and the metadata is what makes citing a source possible
  at the far end of the pipeline
- Metadata travels with the text all the way through the pipeline
- `RecursiveCharacterTextSplitter` breaks at the most natural place available
- Chunk size is a real trade-off with no universal answer: too small loses context, too
  large brings back mostly noise. Start at 500-1000 and measure
- Overlap (10-20%) fixes the boundary problem, which no chunk size can avoid

## Next session

We have pieces. Now: given a question, **which pieces are relevant?**

Word-matching can't do it — "heaviest thing they can fly" and "no payload exceeds 2.8 kg"
share no words at all. Next session we turn text into numbers that capture *meaning*, and
suddenly it can.